In [ ]:
# ANOVA on statistically significant differences between classes across layers
from scipy import stats
import pandas as pd

def layer_anova_analysis(features_dict, labels, task_labels):
    """
    Perform ANOVA analysis on each layer's features.
    
    Args:
        features_dict: Dictionary with layer names as keys and PCA-transformed features as values
        labels: Original class labels (music genre or speech type)
        task_labels: Binary labels (0=music, 1=speech)
    
    Returns:
        DataFrame with ANOVA results for each layer
    """
    results = []
    
    for layer_name, features_pca in features_dict.items():
        # For overall task discrimination (music vs speech)
        f_val_task, p_val_task = stats.f_oneway(
            features_pca[task_labels == 0, :2].T,  # Music samples, first 2 PCs
            features_pca[task_labels == 1, :2].T   # Speech samples, first 2 PCs
        )
        
        # Separation within music classes
        music_samples = features_pca[task_labels == 0]
        music_labels = labels[task_labels == 0]
        music_groups = [music_samples[music_labels == label, :2].T 
                       for label in np.unique(music_labels)]
        f_val_music, p_val_music = stats.f_oneway(*music_groups)
        
        # Separation within speech classes
        speech_samples = features_pca[task_labels == 1]
        speech_labels = labels[task_labels == 1]
        speech_groups = [speech_samples[speech_labels == label, :2].T 
                        for label in np.unique(speech_labels)]
        f_val_speech, p_val_speech = stats.f_oneway(*speech_groups)
        
        results.append({
            'layer': layer_name,
            'task_f_value': f_val_task,
            'task_p_value': p_val_task,
            'music_f_value': f_val_music,
            'music_p_value': p_val_music,
            'speech_f_value': f_val_speech, 
            'speech_p_value': p_val_speech
        })
    
    return pd.DataFrame(results)

In [ ]:
# add silhouette scores
from sklearn.metrics import silhouette_score

def compute_silhouette_scores(features_dict, task_labels):
    """Compute silhouette scores for task separation at each layer"""
    results = []
    
    for layer_name, features in features_dict.items():
        # Standardize and apply PCA
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(features)
        pca = PCA(n_components=2)
        features_pca = pca.fit_transform(features_scaled)
        
        # Compute silhouette score (how well separated the clusters are)
        sil_score = silhouette_score(features_pca, task_labels)
        
        results.append({
            'layer': layer_name,
            'silhouette_score': sil_score
        })
    
    return pd.DataFrame(results)